[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/38_causal_machine_learning/90_exercises.ipynb)

# Bài tập — Chương 38: Causal Machine Learning (L0 → L7)

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/38_causal_machine_learning

In [ ]:
import warnings

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import KFold

from ml_course.checks import check_close, check_equal, check_greater, check_less, check_true
from ml_course.datasets import make_causal_marketing

warnings.filterwarnings("ignore")
data = make_causal_marketing(20000, seed=1)
FEATURES = ["age", "income_mvnd", "past_purchases"]
X, T, Y, tau = data[FEATURES].values, data.coupon.values, data.spend_kvnd.values, data.true_effect.values
ATE = tau.mean()

## L0 — Nhớ khái niệm

**Câu 1.** (a) Confounder, mediator, collider — kiểm soát mỗi loại gây gì? (b) ITE, ATE, ATT, CATE. (c) Ba giả định của suy luận nhân quả từ dữ liệu quan sát. (d) Điểm xu hướng và IPW. (e) "Doubly robust". (f) Vì sao uplift ≠ phản hồi?

✍️ Trả lời:

## L1 — Giải thích bằng lời

**Câu 2.** Một công ty phần mềm thấy khách hàng dùng **gói hỗ trợ cao cấp** có tỉ lệ huỷ thấp hơn 40% so với khách không dùng. Họ đề xuất **tặng gói cao cấp cho mọi khách** để giảm huỷ. Phân tích các lý do khiến kết luận có thể sai, và đề xuất cách xác định tác động thật.

✍️ Trả lời:

## L2 — Tính tay

**Câu 3.** Bảng tỉ lệ khỏi bệnh: nhóm **nhẹ** — thuốc A: 90/100, thuốc B: 640/800; nhóm **nặng** — A: 240/800, B: 20/100. Tính tỉ lệ khỏi **gộp** của A, B (`pooled_A`, `pooled_B`) và **chênh lệch trong từng nhóm** (A − B) `diff_mild`, `diff_severe`.

In [ ]:
### START CODE HERE ### (≈ 4 dòng)
pooled_A, pooled_B = None, None
diff_mild = None
diff_severe = None
### END CODE HERE ###
print(f"gộp: A = {pooled_A:.3f}, B = {pooled_B:.3f} | trong nhóm nhẹ: A − B = {diff_mild:+.3f} | nhóm nặng: {diff_severe:+.3f}")

In [ ]:
check_less(pooled_A, pooled_B, "gộp chung: A trông tệ hơn")
check_close([diff_mild, diff_severe], [0.1, 0.1], message="trong từng nhóm: A tốt hơn 10 điểm % (Simpson)")

**Câu 4.** Bốn người: $(T, Y, e) = (1, 10, 0.5), (1, 6, 0.8), (0, 4, 0.5), (0, 2, 0.2)$. Tính IPW Horvitz–Thompson `ipw_ht` $= \frac1n\sum[TY/e - (1 - T)Y/(1 - e)]$.

In [ ]:
### START CODE HERE ### (≈ 1 dòng)
ipw_ht = None
### END CODE HERE ###
print(ipw_ht)

In [ ]:
check_close(ipw_ht, (20 + 7.5 - 8 - 2.5) / 4, message="IPW Horvitz–Thompson")

## L3 — Code

**Bài 5.** Viết `ipw_ate(Y, T, e, normalized)` (Horvitz–Thompson khi `normalized=False`, Hajek khi `True`) và `smd(x, t, w)` (standardized mean difference có trọng số). Áp dụng với điểm xu hướng logistic trên dữ liệu marketing.

In [ ]:
def ipw_ate(Y, T, e, normalized=True):
    ### START CODE HERE ### (≈ 5 dòng)
    return None
    ### END CODE HERE ###


def smd(x, t, w=None):
    ### START CODE HERE ### (≈ 4 dòng)
    return None
    ### END CODE HERE ###

In [ ]:
e5 = LogisticRegression(max_iter=1000).fit(X, T).predict_proba(X)[:, 1]
est_ht, est_hajek = ipw_ate(Y, T, e5, normalized=False), ipw_ate(Y, T, e5)
print(f"IPW HT = {est_ht:.1f} | Hajek = {est_hajek:.1f} | ATE thật = {ATE:.1f}")
w5 = T / e5 + (1 - T) / (1 - e5)
check_close(ipw_ate(np.array([10, 6, 4, 2.0]), np.array([1, 1, 0, 0]), np.array([0.5, 0.8, 0.5, 0.2]), normalized=False), 4.25, message="khớp câu 4")
check_less(abs(est_hajek - ATE), 8, "Hajek gần ATE")
check_greater(abs(smd(X[:, 2], T)), 0.3, "lịch sử mua mất cân bằng trước khi cân")
check_less(max(abs(smd(X[:, k], T, w5)) for k in range(3)), 0.05, "sau IPW: mọi SMD < 0.05")

**Bài 6.** Viết `aipw_from_components(Y, T, mu1, mu0, e)`. Kiểm tra tính **doubly robust** trên dữ liệu mô phỏng, nơi ta biết mô hình kết quả đúng $\mu_t^*(x)$ và xu hướng đúng $e^*(x)$: thay lần lượt bằng phiên bản **sai**.

In [ ]:
def aipw_from_components(Y, T, mu1, mu0, e):
    ### START CODE HERE ### (≈ 1 dòng)
    return None
    ### END CODE HERE ###

In [ ]:
r6 = np.random.default_rng(6)
n6 = 200000
x6 = r6.normal(size=n6)
e_true = 1 / (1 + np.exp(-1.5 * x6))
t6 = (r6.random(n6) < e_true).astype(int)
mu0_true, mu1_true = 2 * x6, 2 * x6 + 3.0                              # ATE thật = 3
y6 = np.where(t6 == 1, mu1_true, mu0_true) + r6.normal(size=n6)
wrong_mu1, wrong_mu0 = np.full(n6, y6[t6 == 1].mean()), np.full(n6, y6[t6 == 0].mean())   # bỏ qua x
wrong_e = np.full(n6, t6.mean())
cases6 = {"đúng cả hai": (mu1_true, mu0_true, e_true), "kết quả sai": (wrong_mu1, wrong_mu0, e_true),
          "xu hướng sai": (mu1_true, mu0_true, wrong_e), "sai cả hai": (wrong_mu1, wrong_mu0, wrong_e)}
est6 = {k: aipw_from_components(y6, t6, *v) for k, v in cases6.items()}
print({k: round(v, 3) for k, v in est6.items()}, "| naive =", round(y6[t6 == 1].mean() - y6[t6 == 0].mean(), 3))
for k in ["đúng cả hai", "kết quả sai", "xu hướng sai"]:
    check_less(abs(est6[k] - 3.0), 0.05, f"{k}: gần ATE = 3")
check_greater(abs(est6["sai cả hai"] - 3.0), 1.0, "sai cả hai: chệch")

## L4 — Debug

**Bài 7.** Nhà phân tích ước lượng tác động của coupon bằng hồi quy và "cho chắc" thêm mọi biến có trong bảng, gồm cả `visits_after` — **số lần ghé cửa hàng trong tháng sau khi nhận coupon**. Kết quả: coupon gần như không có tác dụng. Tìm lỗi và sửa `adjustment_set`.

In [ ]:
r7 = np.random.default_rng(7)
d7 = make_causal_marketing(20000, seed=3)
visits_after = 2 + 0.02 * d7.true_effect * d7.coupon + 0.3 * d7.past_purchases + r7.normal(0, 0.5, len(d7))   # coupon → ghé nhiều hơn
d7 = d7.assign(visits_after=visits_after)


def effect_with(columns):
    return LinearRegression().fit(d7[["coupon"] + columns].values, d7.spend_kvnd).coef_[0]


def adjustment_set():
    ### START CODE HERE ### (≈ 1 dòng)
    return ["age", "income_mvnd", "past_purchases", "visits_after"]
    ### END CODE HERE ###


buggy7 = effect_with(["age", "income_mvnd", "past_purchases", "visits_after"])
fixed7 = effect_with(adjustment_set())
print(f"kiểm soát cả visits_after: {buggy7:.1f} | chỉ confounder trước điều trị: {fixed7:.1f} | ATE thật = {d7.true_effect.mean():.1f}")

In [ ]:
check_true("visits_after" not in adjustment_set(), "không kiểm soát biến sau điều trị")
check_less(abs(fixed7 - d7.true_effect.mean()), 8, "ước lượng đúng gần ATE")
check_less(buggy7, d7.true_effect.mean() - 40, "tái hiện: kiểm soát mediator làm ước lượng chệch xuống")

**Lưu ý dữ liệu:** trong mô phỏng này `spend` không phụ thuộc trực tiếp `visits_after`, nhưng `visits_after` mang thông tin về hiệu ứng (nó tăng theo `true_effect × coupon`) — một **hệ quả** của điều trị.

## L5 — Phân tích

**Bài 8.** **Confounder không đo được.** Mô phỏng: $U \sim\mathcal N(0, 1)$ không quan sát được, $X$ quan sát được; $T \sim \text{Bernoulli}(\sigma(0.5X + \gamma U))$; $Y = 2T + X + \gamma U + \varepsilon$ (ATE thật = 2). Với $\gamma \in \{0, 0.5, 1, 2\}$: ước lượng bằng OLS điều chỉnh $X$ (không có $U$) kèm khoảng tin cậy bootstrap 95% (100 lần); lưu `res8[γ] = (ước lượng, cận dưới, cận trên)`. Nhận xét.

In [ ]:
res8 = {}
### START CODE HERE ### (≈ 14 dòng)
# với γ: sinh 5 000 mẫu (seed 8); OLS Y ~ T + X; bootstrap 100 lần cho KTC
pass
### END CODE HERE ###

In [ ]:
check_less(abs(res8[0.0][0] - 2), 0.1, "γ = 0: chỉ sai lệch nhỏ do mẫu")
check_greater(res8[2.0][0] - 2, 1.0, "γ lớn: chệch nặng")
check_true(not (res8[2.0][1] <= 2 <= res8[2.0][2]), "γ lớn: KTC hẹp nhưng KHÔNG chứa giá trị thật")
check_greater(res8[2.0][0], res8[0.5][0], "chệch tăng theo sức mạnh confounder")

## L6 — Thực tế

**Bài 9.** **Phân tích A/B test với CUPED.** Thí nghiệm phát coupon ngẫu nhiên (`make_causal_marketing(4000, seed=9, randomized=True)`). (1) Ước lượng ATE bằng chênh lệch trung bình hai nhóm và sai số chuẩn → `ate_raw`, `se_raw`. (2) **CUPED** (Deng et al., 2013 — dùng ở Microsoft, Booking...): dùng biến **trước thí nghiệm** tương quan với kết quả (ở đây `income_mvnd`, `past_purchases`) để giảm phương sai: $Y^{\text{cuped}} = Y - \theta^\top(X - \bar X)$, $\theta$ hồi quy $Y$ theo $X$ trên toàn mẫu (không dùng $T$); rồi chênh lệch trung bình của $Y^{\text{cuped}}$ → `ate_cuped`, `se_cuped`.

In [ ]:
ab9 = make_causal_marketing(4000, seed=9, randomized=True)
Y9, T9, X9 = ab9.spend_kvnd.values, ab9.coupon.values, ab9[["income_mvnd", "past_purchases"]].values


def diff_in_means(y, t):
    diff = y[t == 1].mean() - y[t == 0].mean()
    se = np.sqrt(y[t == 1].var(ddof=1) / (t == 1).sum() + y[t == 0].var(ddof=1) / (t == 0).sum())
    return diff, se


### START CODE HERE ### (≈ 4 dòng)
ate_raw, se_raw, ate_cuped, se_cuped = None, None, None, None
### END CODE HERE ###
print(f"thường: {ate_raw:.1f} ± {1.96 * se_raw:.1f} | CUPED: {ate_cuped:.1f} ± {1.96 * se_cuped:.1f} | ATE thật = {ab9.true_effect.mean():.1f}")
print(f"CUPED giảm sai số chuẩn {1 - se_cuped / se_raw:.0%} ≈ cần ít hơn {1 - (se_cuped / se_raw) ** 2:.0%} số mẫu cho cùng độ chính xác")

In [ ]:
check_less(se_cuped, 0.6 * se_raw, "CUPED giảm sai số chuẩn mạnh")
check_less(abs(ate_cuped - ab9.true_effect.mean()), 3 * se_cuped + 5, "CUPED vẫn không chệch")
check_less(abs(ate_raw - ab9.true_effect.mean()), 3 * se_raw, "ước lượng thường không chệch (nhưng rộng hơn)")

## L7 — Nâng cao

**Bài 10.** **Double Machine Learning** (Chernozhukov et al., 2018) cho mô hình tuyến tính một phần $Y = \theta T + g(X) + \varepsilon$, $T = m(X) + \nu$ với $g$, $m$ **phi tuyến**. Cài `dml_plr(X, T, Y)`: cross-fitting 5 fold — học $\hat\ell(x) \approx \mathbb E[Y\mid x]$ và $\hat m(x) \approx \mathbb E[T\mid x]$ bằng GBM trên các fold khác, tính phần dư $\tilde Y = Y - \hat\ell(X)$, $\tilde T = T - \hat m(X)$, rồi $\hat\theta = \sum\tilde T\tilde Y/\sum\tilde T^2$. So với OLS tuyến tính $Y\sim T + X$.

In [ ]:
r10 = np.random.default_rng(10)
n10 = 8000
X10 = r10.uniform(-2, 2, (n10, 4))
m_x = 1.5 * np.sin(2 * X10[:, 0]) + X10[:, 1] ** 2 - 1                # confounding phi tuyến
T10 = m_x + r10.normal(size=n10)                                       # điều trị liên tục (vd mức giảm giá)
g_x = 3 * np.sin(2 * X10[:, 0]) + 2 * X10[:, 1] ** 2 + X10[:, 2]
Y10 = 1.0 * T10 + g_x + r10.normal(size=n10)                           # θ thật = 1


def dml_plr(X, T, Y, n_folds=5):
    ### START CODE HERE ### (≈ 9 dòng)
    return None
    ### END CODE HERE ###


theta_ols = LinearRegression().fit(np.column_stack([T10, X10]), Y10).coef_[0]
theta_dml, se_dml = dml_plr(X10, T10, Y10)
print(f"OLS tuyến tính: θ = {theta_ols:.3f} | DML: θ = {theta_dml:.3f} ± {1.96 * se_dml:.3f} | θ thật = 1.000")

In [ ]:
check_greater(abs(theta_ols - 1.0), 0.2, "OLS tuyến tính chệch khi confounding phi tuyến")
check_less(abs(theta_dml - 1.0), 0.05, "DML gần θ thật")
check_true(theta_dml - 1.96 * se_dml - 0.01 <= 1.0 <= theta_dml + 1.96 * se_dml + 0.01, "KTC của DML chứa θ thật")